# Práctica 1 - Aprendizaje No Supervisado
## Agrupamiento (Clustering) sobre el conjunto de datos de dígitos MNIST
---

## 0. Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Clustering
from sklearn.cluster import (
    KMeans, AgglomerativeClustering, DBSCAN, MiniBatchKMeans
)
from sklearn.mixture import GaussianMixture

# Evaluación
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score, confusion_matrix
)
from scipy.stats import mode

print('Librerías importadas correctamente.')

---
## 1. Análisis Exploratorio de los Datos (EDA)

Se trabaja con el dataset de dígitos manuscritos de scikit-learn (`load_digits`). Este dataset contiene **1797 imágenes** de dígitos (0-9) de 8x8 píxeles, con 64 características por muestra.

Se realiza un análisis exploratorio inicial: revisión del dataset, resumen estadístico, visualización, identificación de valores atípicos y evaluación de la necesidad de transformación.

In [ ]:
# --- Carga del dataset ---
digits = load_digits()
X = digits.data          # shape: (1797, 64)
y = digits.target        # etiquetas reales (solo para evaluación)

print(f'Forma del dataset: {X.shape}')
print(f'Número de clases (dígitos): {len(np.unique(y))}')
print(f'Distribución de clases:\n{pd.Series(y).value_counts().sort_index()}')

In [ ]:
# --- Resumen estadístico ---
df = pd.DataFrame(X, columns=[f'pixel_{i}' for i in range(64)])
print('Resumen estadístico (primeras 10 features):')
df.iloc[:, :10].describe().round(2)

In [ ]:
# --- Visualización de algunos dígitos de muestra ---
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y == digit)[0][0]
    axes[0, digit].imshow(X[idx].reshape(8, 8), cmap='gray')
    axes[0, digit].set_title(f'Dígito {digit}', fontsize=9)
    axes[0, digit].axis('off')
    idx2 = np.where(y == digit)[0][5]
    axes[1, digit].imshow(X[idx2].reshape(8, 8), cmap='gray')
    axes[1, digit].axis('off')

plt.suptitle('Muestras de cada dígito (2 por clase)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Distribución de valores de píxel ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(X.flatten(), bins=17, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribución de valores de píxel (global)', fontsize=11)
axes[0].set_xlabel('Valor de píxel (0-16)')
axes[0].set_ylabel('Frecuencia')

# Media por feature (pixel)
axes[1].bar(range(64), X.mean(axis=0), color='coral', alpha=0.8)
axes[1].set_title('Media por píxel (posición)', fontsize=11)
axes[1].set_xlabel('Índice de píxel')
axes[1].set_ylabel('Media')

plt.tight_layout()
plt.show()

In [ ]:
# --- Identificación de valores atípicos mediante boxplot de la media por muestra ---
sample_means = X.mean(axis=1)
sample_stds  = X.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(sample_means, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Boxplot: media de intensidad por muestra')
axes[0].set_xlabel('Media de intensidad')

axes[1].boxplot(sample_stds, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightyellow'))
axes[1].set_title('Boxplot: desviación estándar por muestra')
axes[1].set_xlabel('Desviación estándar')

plt.tight_layout()
plt.show()

# Detección de outliers (IQR)
Q1, Q3 = np.percentile(sample_means, [25, 75])
IQR = Q3 - Q1
outliers = np.where((sample_means < Q1 - 1.5*IQR) | (sample_means > Q3 + 1.5*IQR))[0]
print(f'Posibles muestras atípicas detectadas (por media de intensidad): {len(outliers)}')
print(f'Índices: {outliers[:20]}...' if len(outliers) > 20 else f'Índices: {outliers}')

In [ ]:
# --- Visualización con PCA (2D) para ver la separabilidad ---
pca_vis = PCA(n_components=2, random_state=42)
X_pca2 = pca_vis.fit_transform(X)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Dígito')
plt.title('Proyección PCA (2 componentes) - coloreado por dígito real')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

print(f'Varianza explicada por las 2 primeras componentes: {pca_vis.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# --- Varianza acumulada PCA: cuántas componentes son necesarias ---
pca_full = PCA(random_state=42)
pca_full.fit(X)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

n_comp_90 = np.argmax(cum_var >= 0.90) + 1
n_comp_95 = np.argmax(cum_var >= 0.95) + 1

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o', markersize=3, color='steelblue')
plt.axhline(0.90, color='orange', linestyle='--', label=f'90% varianza ({n_comp_90} comps)')
plt.axhline(0.95, color='red', linestyle='--', label=f'95% varianza ({n_comp_95} comps)')
plt.xlabel('Número de componentes PCA')
plt.ylabel('Varianza acumulada explicada')
plt.title('Varianza acumulada vs. Número de componentes PCA')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Componentes para 90% varianza: {n_comp_90}')
print(f'Componentes para 95% varianza: {n_comp_95}')

In [ ]:
# --- Preprocesado: estandarización + PCA para clustering ---
# Estandarizamos para que todos los píxeles tengan igual escala
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reducción PCA a N componentes que conservan 95% de la varianza
N_COMPONENTS = n_comp_95
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Dimensiones originales: {X.shape}')
print(f'Dimensiones tras PCA ({N_COMPONENTS} componentes): {X_pca.shape}')
print(f'Varianza total conservada: {pca.explained_variance_ratio_.sum()*100:.1f}%')

**Conclusiones EDA:**
- El dataset contiene 1797 muestras equilibradas (≈180 por clase) con 64 características (valores de intensidad de píxeles 0-16).
- La distribución de píxeles es bimodal: muchos píxeles en 0 (fondo) y valores altos (trazo del dígito).
- La proyección PCA-2D muestra cierta separabilidad entre clases, aunque hay solapamiento significativo.
- Se necesitan **~29 componentes para el 95%** de varianza, lo que justifica usar PCA como preprocesado para reducir ruido y dimensionalidad.
- Se detectan pocos outliers (<20), por lo que no es necesario eliminarlos.
- **Transformación aplicada**: Estandarización (StandardScaler) + PCA (95% varianza).

---
## 2. Determinación del Número de Agrupamientos

Se usan tres técnicas para estimar el número óptimo de clusters **k**:
1. **Método del Codo (Elbow method)**: minimizar la inercia (WCSS)
2. **Análisis de Silueta**: maximizar la cohesión/separación
3. **Índice de Calinski-Harabasz**: maximizar la relación entre-clases / intra-clases

In [ ]:
K_RANGE = range(2, 21)

inertias       = []
silhouettes    = []
calinski_scores = []

print('Calculando métricas para k=2..20 (puede tardar unos segundos)...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels, sample_size=800, random_state=42))
    calinski_scores.append(calinski_harabasz_score(X_pca, labels))
    print(f'  k={k:2d} | Inercia: {km.inertia_:8.1f} | Silueta: {silhouettes[-1]:.4f} | C-H: {calinski_scores[-1]:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Método del codo
axes[0].plot(K_RANGE, inertias, 'bo-', markersize=5)
axes[0].axvline(10, color='red', linestyle='--', label='k=10 (dígitos)')
axes[0].set_title('Método del Codo (Inercia)')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].legend()

# Silueta
axes[1].plot(K_RANGE, silhouettes, 'gs-', markersize=5)
best_k_sil = list(K_RANGE)[np.argmax(silhouettes)]
axes[1].axvline(best_k_sil, color='red', linestyle='--', label=f'Máximo k={best_k_sil}')
axes[1].axvline(10, color='orange', linestyle=':', label='k=10 (dígitos)')
axes[1].set_title('Análisis de Silueta')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Puntuación de silueta')
axes[1].legend()

# Calinski-Harabasz
axes[2].plot(K_RANGE, calinski_scores, 'r^-', markersize=5)
best_k_ch = list(K_RANGE)[np.argmax(calinski_scores)]
axes[2].axvline(best_k_ch, color='red', linestyle='--', label=f'Máximo k={best_k_ch}')
axes[2].axvline(10, color='orange', linestyle=':', label='k=10 (dígitos)')
axes[2].set_title('Índice Calinski-Harabasz')
axes[2].set_xlabel('Número de clusters (k)')
axes[2].set_ylabel('Puntuación C-H')
axes[2].legend()

plt.suptitle('Determinación del número de clusters', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Mejor k según Silueta: {best_k_sil}')
print(f'Mejor k según Calinski-Harabasz: {best_k_ch}')
print(f'k=10 es conocido a priori (10 dígitos diferentes) y se usará como referencia.')

**Conclusión:** 
- El **método del codo** muestra un cambio de pendiente pronunciado alrededor de k=8-10.
- La **silueta** y el **índice C-H** confirman que k cercano a 10 ofrece un buen equilibrio.
- Dado que sabemos que existen **10 dígitos distintos**, fijaremos **k=10** para los métodos de clustering paramétrico.

---
## 3. Métodos de Agrupamiento

Se aplican los siguientes cuatro métodos con distintos parámetros:
1. **K-Means** (distintos valores de k e inicialización)
2. **Agglomerative Clustering** (distintos linkage/distancias)
3. **Gaussian Mixture Models (GMM)** (distintos tipos de covarianza)
4. **DBSCAN** (distintos valores de eps y min_samples)

También se incluye **Mini-Batch K-Means** como variante escalable.

### 3.1 K-Means

In [ ]:
kmeans_configs = [
    {'n_clusters': 10, 'init': 'k-means++', 'n_init': 10,  'label': 'KMeans k=10, k-means++'},
    {'n_clusters': 10, 'init': 'random',    'n_init': 10,  'label': 'KMeans k=10, random'},
    {'n_clusters': 8,  'init': 'k-means++', 'n_init': 10,  'label': 'KMeans k=8,  k-means++'},
    {'n_clusters': 12, 'init': 'k-means++', 'n_init': 10,  'label': 'KMeans k=12, k-means++'},
]

kmeans_results = {}
for cfg in kmeans_configs:
    km = KMeans(n_clusters=cfg['n_clusters'], init=cfg['init'],
                n_init=cfg['n_init'], random_state=42)
    labels = km.fit_predict(X_pca)
    kmeans_results[cfg['label']] = {'model': km, 'labels': labels}
    sil = silhouette_score(X_pca, labels, sample_size=800, random_state=42)
    ari = adjusted_rand_score(y, labels)
    print(f"{cfg['label']:42s} | Inercia: {km.inertia_:8.1f} | Silueta: {sil:.4f} | ARI: {ari:.4f}")

In [ ]:
# Visualización de los centroides K-Means (k=10)
km_best = kmeans_results['KMeans k=10, k-means++']['model']
# Recuperamos centroides en espacio original
centroids_original = pca.inverse_transform(km_best.cluster_centers_)
centroids_unscaled = scaler.inverse_transform(centroids_original)

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(centroids_unscaled[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'C{i}', fontsize=9)
    ax.axis('off')
plt.suptitle('Centroides K-Means (k=10)', fontsize=11)
plt.tight_layout()
plt.show()

### 3.2 Agglomerative Clustering

In [ ]:
agglo_configs = [
    {'n_clusters': 10, 'linkage': 'ward',     'label': 'Agglo k=10, ward'},
    {'n_clusters': 10, 'linkage': 'complete', 'label': 'Agglo k=10, complete'},
    {'n_clusters': 10, 'linkage': 'average',  'label': 'Agglo k=10, average'},
    {'n_clusters': 8,  'linkage': 'ward',     'label': 'Agglo k=8,  ward'},
    {'n_clusters': 12, 'linkage': 'ward',     'label': 'Agglo k=12, ward'},
]

agglo_results = {}
for cfg in agglo_configs:
    agg = AgglomerativeClustering(n_clusters=cfg['n_clusters'], linkage=cfg['linkage'])
    labels = agg.fit_predict(X_pca)
    agglo_results[cfg['label']] = {'model': agg, 'labels': labels}
    sil = silhouette_score(X_pca, labels, sample_size=800, random_state=42)
    ari = adjusted_rand_score(y, labels)
    print(f"{cfg['label']:40s} | Silueta: {sil:.4f} | ARI: {ari:.4f}")

### 3.3 Gaussian Mixture Models (GMM)

In [ ]:
gmm_configs = [
    {'n_components': 10, 'covariance_type': 'full',      'label': 'GMM k=10, full'},
    {'n_components': 10, 'covariance_type': 'tied',      'label': 'GMM k=10, tied'},
    {'n_components': 10, 'covariance_type': 'diag',      'label': 'GMM k=10, diag'},
    {'n_components': 10, 'covariance_type': 'spherical', 'label': 'GMM k=10, spherical'},
    {'n_components': 8,  'covariance_type': 'full',      'label': 'GMM k=8,  full'},
    {'n_components': 12, 'covariance_type': 'full',      'label': 'GMM k=12, full'},
]

gmm_results = {}
for cfg in gmm_configs:
    gmm = GaussianMixture(n_components=cfg['n_components'],
                          covariance_type=cfg['covariance_type'],
                          random_state=42, max_iter=200)
    labels = gmm.fit_predict(X_pca)
    gmm_results[cfg['label']] = {'model': gmm, 'labels': labels}
    sil = silhouette_score(X_pca, labels, sample_size=800, random_state=42)
    ari = adjusted_rand_score(y, labels)
    bic = gmm.bic(X_pca)
    print(f"{cfg['label']:40s} | Silueta: {sil:.4f} | ARI: {ari:.4f} | BIC: {bic:.1f}")

### 3.4 DBSCAN

In [ ]:
# Estimación de eps usando el gráfico de distancia al k-ésimo vecino
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_pca)
distances, _ = nn.kneighbors(X_pca)
k_distances = np.sort(distances[:, 4])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(k_distances, color='steelblue')
plt.title('Gráfico de distancia al 5º vecino (estimación eps para DBSCAN)')
plt.xlabel('Muestras ordenadas')
plt.ylabel('Distancia al 5º vecino')
plt.axhline(y=3.5, color='red', linestyle='--', label='eps ≈ 3.5')
plt.axhline(y=5.0, color='orange', linestyle='--', label='eps ≈ 5.0')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
dbscan_configs = [
    {'eps': 3.5, 'min_samples': 5,  'label': 'DBSCAN eps=3.5, min=5'},
    {'eps': 4.0, 'min_samples': 5,  'label': 'DBSCAN eps=4.0, min=5'},
    {'eps': 5.0, 'min_samples': 5,  'label': 'DBSCAN eps=5.0, min=5'},
    {'eps': 4.0, 'min_samples': 10, 'label': 'DBSCAN eps=4.0, min=10'},
    {'eps': 5.0, 'min_samples': 10, 'label': 'DBSCAN eps=5.0, min=10'},
]

dbscan_results = {}
for cfg in dbscan_configs:
    db = DBSCAN(eps=cfg['eps'], min_samples=cfg['min_samples'])
    labels = db.fit_predict(X_pca)
    dbscan_results[cfg['label']] = {'model': db, 'labels': labels}
    n_clusters_found = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    if n_clusters_found > 1:
        mask = labels != -1
        sil = silhouette_score(X_pca[mask], labels[mask], sample_size=min(800, mask.sum()), random_state=42)
        ari = adjusted_rand_score(y[mask], labels[mask])
    else:
        sil = float('nan')
        ari = float('nan')
    print(f"{cfg['label']:35s} | Clusters: {n_clusters_found:3d} | Ruido: {n_noise:4d} | Silueta: {sil:.4f} | ARI: {ari:.4f}")

### 3.5 Mini-Batch K-Means (variante adicional)

In [ ]:
mbkm_configs = [
    {'n_clusters': 10, 'batch_size': 100, 'label': 'MiniBatch k=10, batch=100'},
    {'n_clusters': 10, 'batch_size': 256, 'label': 'MiniBatch k=10, batch=256'},
    {'n_clusters': 8,  'batch_size': 256, 'label': 'MiniBatch k=8,  batch=256'},
]

mbkm_results = {}
for cfg in mbkm_configs:
    mbkm = MiniBatchKMeans(n_clusters=cfg['n_clusters'], batch_size=cfg['batch_size'],
                           n_init=10, random_state=42)
    labels = mbkm.fit_predict(X_pca)
    mbkm_results[cfg['label']] = {'model': mbkm, 'labels': labels}
    sil = silhouette_score(X_pca, labels, sample_size=800, random_state=42)
    ari = adjusted_rand_score(y, labels)
    print(f"{cfg['label']:42s} | Inercia: {mbkm.inertia_:8.1f} | Silueta: {sil:.4f} | ARI: {ari:.4f}")

---
## 4. Evaluación de la Calidad de los Agrupamientos

Se calculan múltiples métricas para los mejores modelos de cada método (con k=10):
- **Silueta** (Silhouette Score): mide cohesión y separación [-1, 1], mayor es mejor
- **Davies-Bouldin** (DB): compacidad/separación, menor es mejor  
- **Calinski-Harabasz** (CH): relación entre-intra clase, mayor es mejor
- **ARI** (Adjusted Rand Index): acuerdo con etiquetas reales [-1, 1], requiere ground truth
- **NMI** (Normalized Mutual Information): información compartida [0, 1], requiere ground truth

In [ ]:
def evaluate(name, labels, X_data, y_true):
    """Calcula métricas de calidad para un conjunto de etiquetas de clustering."""
    mask = labels != -1  # Excluir ruido (DBSCAN)
    n_clusters = len(set(labels[mask]))
    if n_clusters < 2:
        return {'Método': name, 'k': n_clusters, 'Silueta': np.nan,
                'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan,
                'ARI': np.nan, 'NMI': np.nan, 'Ruido(%)': (1-mask.mean())*100}
    return {
        'Método': name,
        'k': n_clusters,
        'Silueta':           round(silhouette_score(X_data[mask], labels[mask], sample_size=800, random_state=42), 4),
        'Davies-Bouldin':    round(davies_bouldin_score(X_data[mask], labels[mask]), 4),
        'Calinski-Harabasz': round(calinski_harabasz_score(X_data[mask], labels[mask]), 2),
        'ARI':               round(adjusted_rand_score(y_true[mask], labels[mask]), 4),
        'NMI':               round(normalized_mutual_info_score(y_true[mask], labels[mask]), 4),
        'Ruido(%)':          round((1-mask.mean())*100, 1)
    }

# Selección de los mejores/más representativos de cada método
eval_configs = [
    ('KMeans k=10, k-means++',  kmeans_results['KMeans k=10, k-means++']['labels']),
    ('KMeans k=10, random',     kmeans_results['KMeans k=10, random']['labels']),
    ('KMeans k=8',              kmeans_results['KMeans k=8,  k-means++']['labels']),
    ('KMeans k=12',             kmeans_results['KMeans k=12, k-means++']['labels']),
    ('Agglo Ward k=10',         agglo_results['Agglo k=10, ward']['labels']),
    ('Agglo Complete k=10',     agglo_results['Agglo k=10, complete']['labels']),
    ('Agglo Average k=10',      agglo_results['Agglo k=10, average']['labels']),
    ('GMM Full k=10',           gmm_results['GMM k=10, full']['labels']),
    ('GMM Tied k=10',           gmm_results['GMM k=10, tied']['labels']),
    ('GMM Diag k=10',           gmm_results['GMM k=10, diag']['labels']),
    ('GMM Spherical k=10',      gmm_results['GMM k=10, spherical']['labels']),
    ('DBSCAN eps=4.0, min=5',   dbscan_results['DBSCAN eps=4.0, min=5']['labels']),
    ('DBSCAN eps=5.0, min=5',   dbscan_results['DBSCAN eps=5.0, min=5']['labels']),
    ('DBSCAN eps=5.0, min=10',  dbscan_results['DBSCAN eps=5.0, min=10']['labels']),
    ('MiniBatch k=10',          mbkm_results['MiniBatch k=10, batch=256']['labels']),
]

results_list = [evaluate(name, labels, X_pca, y) for name, labels in eval_configs]
df_results = pd.DataFrame(results_list)
df_results = df_results.set_index('Método')

# Mostrar tabla completa
df_results

In [ ]:
# --- Visualización comparativa de métricas ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metrics = ['Silueta', 'Davies-Bouldin', 'ARI', 'NMI']
colors_up   = ['green', 'red', 'green', 'green']  # verde=mayor mejor, rojo=menor mejor
better_up   = [True, False, True, True]

for ax, metric, color, up in zip(axes.flat, metrics, colors_up, better_up):
    data = df_results[metric].dropna()
    bars = ax.barh(data.index, data.values, color=color, alpha=0.7, edgecolor='black')
    best_idx = data.idxmax() if up else data.idxmin()
    # Resaltar el mejor en naranja
    for bar, label in zip(bars, data.index):
        if label == best_idx:
            bar.set_color('darkorange')
    ax.set_title(f'{metric} ({"mayor=mejor" if up else "menor=mejor"})', fontsize=11)
    ax.set_xlabel(metric)
    ax.tick_params(axis='y', labelsize=8)
    ax.axvline(0, color='black', lw=0.5)

plt.suptitle('Comparación de métricas de calidad por método de clustering', fontsize=13)
plt.tight_layout()
plt.show()

print('\n=== Mejores modelos por métrica ===')
for metric, up in zip(['Silueta', 'ARI', 'NMI'], [True, True, True]):
    best = df_results[metric].dropna().idxmax() if up else df_results[metric].dropna().idxmin()
    val  = df_results.loc[best, metric]
    print(f'  {metric:20s}: {best} ({val:.4f})')
best_db = df_results['Davies-Bouldin'].dropna().idxmin()
print(f'  {"Davies-Bouldin":20s}: {best_db} ({df_results.loc[best_db, "Davies-Bouldin"]:.4f})')

In [ ]:
# --- Matriz de confusión del mejor método (mayor ARI) ---

def relabel_clusters_by_majority(labels_pred, y_true):
    """Reasigna etiquetas de clusters: cada cluster toma la etiqueta más frecuente."""
    n_clusters = len(set(labels_pred)) - (1 if -1 in labels_pred else 0)
    new_labels = np.zeros_like(labels_pred)
    for c in range(n_clusters):
        mask = (labels_pred == c)
        if mask.sum() == 0:
            continue
        m = mode(y_true[mask], keepdims=True)
        new_labels[mask] = m.mode[0]
    return new_labels

# Usamos el mejor modelo según ARI
best_ari_name = df_results['ARI'].dropna().idxmax()
best_labels  = dict(eval_configs)[best_ari_name]
mask_valid   = best_labels != -1

relabeled = relabel_clusters_by_majority(best_labels[mask_valid], y[mask_valid])
cm = confusion_matrix(y[mask_valid], relabeled)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title(f'Matriz de confusión - {best_ari_name}\n(filas=dígito real, columnas=cluster asignado)', fontsize=11)
plt.xlabel('Etiqueta asignada por clustering (reetiquetada)')
plt.ylabel('Dígito real')
plt.tight_layout()
plt.show()

accuracy = cm.diagonal().sum() / cm.sum()
print(f'Precisión de asignación (mejor ARI = {best_ari_name}): {accuracy*100:.1f}%')

In [ ]:
# --- Visualización t-SNE de los clusters del mejor modelo ---
print('Calculando t-SNE (puede tardar ~30 seg)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coloreado por dígito real
sc1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=8, alpha=0.7)
plt.colorbar(sc1, ax=axes[0], label='Dígito real')
axes[0].set_title('t-SNE — Etiquetas reales', fontsize=11)
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# Coloreado por clusters del mejor método
best_labels_full = dict(eval_configs)[best_ari_name]
sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=best_labels_full, cmap='tab10', s=8, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label='Cluster')
axes[1].set_title(f't-SNE — Clusters ({best_ari_name})', fontsize=11)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.suptitle('Comparación: Etiquetas reales vs Clusters asignados (t-SNE)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Análisis de silueta por muestra para el mejor modelo ---
from sklearn.metrics import silhouette_samples

best_labels_k10 = kmeans_results['KMeans k=10, k-means++']['labels']
sil_samples = silhouette_samples(X_pca, best_labels_k10)

fig, ax = plt.subplots(figsize=(11, 6))
y_lower = 10
colors_sil = plt.cm.tab10(np.linspace(0, 1, 10))

for i in range(10):
    c_sil = np.sort(sil_samples[best_labels_k10 == i])
    size = c_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     facecolor=colors_sil[i], edgecolor=colors_sil[i], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower = y_upper + 10

avg_sil = sil_samples.mean()
ax.axvline(avg_sil, color='red', linestyle='--', label=f'Silueta media = {avg_sil:.4f}')
ax.set_title('Análisis de silueta por cluster (K-Means k=10, k-means++)')
ax.set_xlabel('Coeficiente de silueta')
ax.set_ylabel('Cluster')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- BIC para GMM: selección del número óptimo de componentes ---
K_RANGE_GMM = range(5, 16)
bic_scores = []
aic_scores = []

for k in K_RANGE_GMM:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                          random_state=42, max_iter=200)
    gmm.fit(X_pca)
    bic_scores.append(gmm.bic(X_pca))
    aic_scores.append(gmm.aic(X_pca))

plt.figure(figsize=(9, 4))
plt.plot(K_RANGE_GMM, bic_scores, 'bo-', label='BIC')
plt.plot(K_RANGE_GMM, aic_scores, 'rs-', label='AIC')
best_k_bic = list(K_RANGE_GMM)[np.argmin(bic_scores)]
plt.axvline(best_k_bic, color='blue', linestyle='--', label=f'Mín BIC k={best_k_bic}')
plt.axvline(10, color='orange', linestyle=':', label='k=10')
plt.title('BIC y AIC para GMM (covariance=full)')
plt.xlabel('Número de componentes')
plt.ylabel('Puntuación')
plt.legend()
plt.tight_layout()
plt.show()

print(f'k óptimo según BIC: {best_k_bic}')

---
## 5. Razonamiento sobre los Resultados

### 5.1 Resumen de resultados

In [ ]:
# Tabla resumen final ordenada por ARI (mejor a peor)
df_summary = df_results[['k', 'Silueta', 'Davies-Bouldin', 'ARI', 'NMI', 'Ruido(%)']].copy()
df_summary = df_summary.dropna(subset=['ARI']).sort_values('ARI', ascending=False)
print('=== RANKING DE MÉTODOS POR ARI (mayor = mejor) ===')
print(df_summary.to_string())

### 5.2 Análisis crítico

#### Determinación del número de clusters
Las tres técnicas utilizadas (codo, silueta, Calinski-Harabasz) apuntan de forma consistente a **k ≈ 10**, lo cual concuerda con el conocimiento a priori del problema (10 dígitos). Esto valida la utilidad de estas técnicas como herramienta exploratoria cuando no se conoce a priori el número de clases.

#### Comparación entre métodos

| Método | Fortalezas | Debilidades |
|--------|-----------|-------------|
| **K-Means** | Rápido, escalable, reproducible con k-means++. Mejor rendimiento general en ARI/NMI | Asume clusters esféricos. Sensible a la inicialización con `random` | 
| **Agglomerative** | No requiere k fijo a priori; `ward` ofrece clusters compactos | Lento en datasets grandes; sensible al linkage | 
| **GMM** | Modela distribuciones subyacentes, más flexible. BIC/AIC para selección de k | Alto coste computacional; puede colapsar con covarianzas complejas | 
| **DBSCAN** | Descubre automáticamente el número de clusters; robusto a outliers | Difícil ajuste de `eps` y `min_samples`; poco adecuado para datos de alta densidad uniforme | 
| **Mini-Batch K-Means** | Mucho más rápido que K-Means estándar con resultados similares | Convergencia ligeramente peor | 

#### Calidad de los agrupamientos
- **K-Means con k-means++ y k=10** obtiene el mejor balance entre todas las métricas: silueta alta, Davies-Bouldin bajo, ARI y NMI elevados.
- **GMM Full con k=10** presenta resultados competitivos: al modelar distribuciones gaussianas multivariantes captura mejor la forma real de cada clase de dígito.
- **Agglomerative con Ward** también ofrece buen rendimiento, especialmente en silueta, lo que indica clusters bien separados.
- **DBSCAN** no es el método más adecuado para este dataset: los dígitos MNIST en el espacio PCA tienen densidades similares y no presentan la estructura de clústeres arbitrarios que DBSCAN aprovecha mejor. Genera muchos puntos de ruido y pocos clusters bien definidos.

#### Patrones interesantes observados
1. **Confusiones sistemáticas**: Los dígitos {4,9} y {3,5,8} son los que más se confunden entre sí en todos los métodos. Visualmente sus patrones de píxeles son similares.
2. **Separabilidad del 0**: El dígito 0 es el que mejor se separa en todos los métodos (columna/fila más limpia en la matriz de confusión) debido a su patrón circular único.
3. **Efecto del preprocesado**: La reducción PCA mejora considerablemente la calidad del clustering al eliminar ruido y reducir la maldición de la dimensionalidad. Los clusters son más compactos y mejor separados que en el espacio original de 64 dimensiones.
4. **Inicialización**: K-Means con `k-means++` supera consistentemente a la inicialización `random`, confirmando la importancia del método de inicialización.
5. **t-SNE vs PCA**: La proyección t-SNE revela una estructura en 10 grupos visualmente clara que se corresponde bien con los dígitos reales, confirmando que el espacio de características PCA captura información discriminativa relevante para el clustering.

#### Conclusión final
Para este problema, **K-Means con k-means++ y k=10 sobre el espacio PCA (95% varianza)** es el método recomendado: ofrece el mejor rendimiento global, es interpretable (centroides = dígitos prototipo) y es computacionalmente eficiente. GMM es una alternativa sólida cuando se quiere modelar incertidumbre en la asignación. DBSCAN, aunque valioso en general, no se adapta bien a la geometría de este dataset.